# Using FlexGen to Offload OPT Models' Weights to RAM and Disk
This notebook is a companion of chapter 9 of the "Domain Specific LLMs in Action" book, author Guglielmo Iozzia, [Manning Publications](https://www.manning.com/), 2024.  
The code in this notebook is to perform inference with a Meta AI's OPT models, by offloading part of models' weights from VRAM to RAM and/or disk, using the [FlexGen](https://github.com/FMInference/FlexLLMGen/) generation engine programmatically. While the code refers to the [OPT 1.3 B](https://huggingface.co/facebook/opt-1.3b) model, the same applies to any other model from the same family. It requires hardware acceleration to be executed.  
More details about the code can be found in the related book's chapter.

Install the FlexGen from source.

In [1]:
!git clone https://github.com/FMInference/FlexLLMGen.git
%cd FlexLLMGen
!pip install -e .

Cloning into 'FlexLLMGen'...
remote: Enumerating objects: 4553, done.
remote: Total 4553 (delta 0), reused 0 (delta 0), pack-reused 4553 (from 1)
Receiving objects: 100% (4553/4553), 38.11 MiB | 12.17 MiB/s, done.
Resolving deltas: 100% (1320/1320), done.
/content/FlexLLMGen
Obtaining file:///content/FlexLLMGen
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for flexllmgen (pyproject.toml) ... done
  Created wheel for flexllmgen: filename=flexllmgen-0.1.7-0.editable-py3-none-any.whl size=12258 sha256=079a76092f55dd0872d4ca3e0d2f1135bb506a2aafa35dd595735ce3954595db
  Stored in directory: /tmp/pip-ephem-wheel-cache-l1hhb9fo/wheels/de/ea/02/63a83fe5d7efc9c3c06b4b479c501b67918329a06fa5559a71
Successfully built flexllmgen


# Using FlexGen and the Transformers library programmatically

Import the required FlexGen classes.

In [4]:
from flexllmgen.flex_opt import (Policy, OptLM, ExecutionEnv, CompressionConfig,
        str2bool)

FlexGen is a high-throughput generation engine designed to run large language models (LLMs) on consumer-grade hardware by utilizing a **three-tier hierarchical memory system**: GPU, CPU, and Disk.

Here is a breakdown of those specific classes and how they orchestrate memory offloading:

### `Policy`
This class acts as the "brain" of the execution strategy. It defines how the model’s tensors (weights, hidden states, and KV cache) are distributed across the GPU, CPU, and Disk.
* **Memory Allocation:** It specifies the percentage of each tensor type to store in each memory tier.
* **Overlap Settings:** It controls whether to overlap compute with memory transfers to hide latency.
* **Batching:** It manages the `gpu_batch_size` and `num_gpu_batches` to maximize throughput.

### `OptLM`
This is the core model wrapper for the OPT (Open Pre-trained Transformer) architecture.
* **Structure:** It initializes the transformer layers and manages the loading of weights from the storage medium.
* **Execution Logic:** Instead of a standard forward pass, `OptLM` implements the FlexGen-specific execution flow, which processes the model layer-by-layer or block-by-block based on the `Policy`.
* **Interface:** It provides the `.generate()` methods used to produce text.

### `ExecutionEnv`
This class initializes and manages the hardware resources available for the inference task.
* **Device Management:** It handles the initialization of the `nvme` (disk), `cpu` (RAM), and `gpu` (VRAM) handles.
* **Resource Tracking:** It keeps track of available memory overhead to prevent "Out of Memory" (OOM) errors during the massive offloading processes.

### `CompressionConfig`
FlexGen achieves its efficiency partly through extreme compression. This class defines the parameters for that compression.
* **Quantization:** It specifies whether to use 4-bit or 8-bit quantization for weights and the KV cache.
* **Group Size:** It defines the granularity of the quantization (e.g., group-wise scaling) to maintain model accuracy despite high compression ratios.

### `str2bool`
A simple but essential utility function.
* **Functionality:** It converts command-line string arguments (like "true", "yes", "1", "false") into actual Python Boolean objects. It is used extensively in the FlexGen scripts to parse user configurations from the terminal.


Download the OPT 1.3 B tokenizer form the Hugging Face's Hub.

In [3]:
from transformers import AutoTokenizer

model_id = "facebook/opt-1.3b"
tokenizer = AutoTokenizer.from_pretrained(model_id, padding_side="left")
tokenizer.add_bos_token = False
stop = tokenizer("\n").input_ids[0]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/653 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/685 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/441 [00:00<?, ?B/s]

Setup the FlexGen execution environment.

In [5]:
offload_dir = './flexgen_offload'
env = ExecutionEnv.create(offload_dir)

Prepare a list of prompts for batch inference.

In [6]:
prompts = [
    "Question: Where were the 2004 Olympics held?\n"
    "Answer: Athens, Greece\n"
    "Question: What is the longest river on the earth?\n"
    "Answer:",

    "Extract the airport codes from this text.\n"
    "Text: \"I want a flight from New York to San Francisco.\"\n"
    "Airport codes: JFK, SFO.\n"
    "Text: \"I want you to book a flight from Phoenix to Las Vegas.\"\n"
    "Airport codes:",
]

Setup an offloading policy.

In [7]:
policy = Policy(len(prompts), 1,
                70, 30, 70, 30, 100, 0,
                overlap=True, sep_layer=True, pin_weight=True,
                cpu_cache_compute=True, attn_sparsity=1.0,
                compress_weight=True,
                comp_weight_config=CompressionConfig(
                    num_bits=4, group_size=64,
                    group_dim=0, symmetric=False),
                compress_cache=False, # Set compress_cache to False
                comp_cache_config=CompressionConfig(
                    num_bits=4, group_size=64,
                    group_dim=2, symmetric=False)
                )

The `Policy` class in FlexGen is essentially a configuration blueprint that tells the engine exactly how to slice and dice the model's memory footprint to fit your hardware.

Here is a breakdown of what those specific settings are doing:

### 1. Throughput & Batching
* **`len(prompts), 1`**: These are the `gpu_batch_size` and `num_gpu_batches`. By setting the batch size to the length of your prompts, you are telling FlexGen to process all inputs in a single parallel sweep on the GPU.

### 2. The "Weight-Cache-Hidden" Offloading Ratios
The six numbers `70, 30, 70, 30, 100, 0` represent the percentage distribution of the three main tensor types across **GPU** and **CPU** (and Disk, implicitly):

* **Weights (70, 30):** 70% of the model weights stay on the **GPU**, while 30% are stored on the **CPU** and swapped in as needed.
* **KV Cache (70, 30):** 70% of the Key-Value cache (context memory) stays on the **GPU**, 30% on the **CPU**.
* **Hidden States (100, 0):** 100% of the intermediate hidden states are kept on the **GPU**. This is common because hidden states are relatively small but accessed frequently.

### 3. Execution Flags
* **`overlap=True`**: Enables "hiding" the time it takes to move data. While the GPU is computing Layer 1, the system is already pre-fetching Layer 2 from the CPU.
* **`sep_layer=True`**: Treats each transformer layer as a separate unit for memory management, which is crucial for the "zigzag" offloading logic FlexGen uses.
* **`pin_weight=True`**: Uses "pinned" (page-locked) CPU memory, which significantly speeds up data transfers between the CPU and GPU.
* **`cpu_cache_compute=True`**: Allows some parts of the KV cache attention mechanism to be computed on the CPU if necessary, rather than moving everything to the GPU.

### 4. Compression Settings (`CompressionConfig`)
This section defines how the model is "shrunk" to save space:

* **`compress_weight=True`**: Enables 4-bit quantization for the weights.
    * **`num_bits=4`**: Reduces weight size by approximately 4x compared to FP16.
    * **`group_size=64`**: Groups 64 constants together for a single scaling factor to minimize accuracy loss.
* **`compress_cache=False`**: You have explicitly turned off compression for the KV cache. Even though a configuration is provided (`num_bits=4` etc.), it is ignored. This ensures maximum precision for the "memory" of the conversation, but at the cost of using much more VRAM/RAM.


Prepare the model to be executed through the FlexGen inference engine and following the preliminary defined offloading policies. This step also downloads the model's checkpoints from the Hugging Face's Hub and manages the conversion process.

In [8]:
path = '~/opt_weights'
model = OptLM(model_id, env, path, policy)

Load the pre-trained pytorch weights of opt-1.3b from huggingface. The downloading and cpu loading can take dozens of minutes. If it seems to get stuck, you can monitor the progress by checking the memory usage of this process.


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]


Convert format:   0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/389 [00:00<?, ?it/s]

  0%|          | 1/389 [00:00<05:13,  1.24it/s]

  8%|▊         | 33/389 [00:00<00:07, 48.40it/s]

 13%|█▎        | 51/389 [00:01<00:05, 58.61it/s]

 17%|█▋        | 65/389 [00:01<00:04, 69.02it/s]

 24%|██▍       | 95/389 [00:01<00:02, 109.50it/s]

 29%|██▉       | 113/389 [00:01<00:02, 115.44it/s]

 33%|███▎      | 129/389 [00:01<00:02, 111.89it/s]

 37%|███▋      | 144/389 [00:01<00:02, 113.87it/s]

 41%|████      | 158/389 [00:01<00:01, 117.82it/s]

 44%|████▍     | 172/389 [00:02<00:02, 96.52it/s] 

 47%|████▋     | 184/389 [00:02<00:02, 81.68it/s]

 50%|████▉     | 194/389 [00:02<00:02, 69.82it/s]

 52%|█████▏    | 203/389 [00:04<00:12, 15.41it/s]

 60%|██████    | 235/389 [00:04<00:04, 31.51it/s]

 69%|██████▊   | 267/389 [00:04<00:02, 51.71it/s]

 74%|███████▎  | 286/389 [00:04<00:01, 64.03it/s]

 78%|███████▊  | 305/389 [00:05<00:01, 69.98it/s]

 85%|████████▌ | 331/389 [00:05<00:0

Generate text for the given set of prompts and then display the generated result for each one.

In [10]:
print("Generate...")
inputs = tokenizer(prompts, padding="max_length", max_length=128)
output_ids = model.generate(
    inputs.input_ids,
    do_sample=True,
    temperature=0.7,
    max_new_tokens=32,
    stop=stop)
outputs = tokenizer.batch_decode(output_ids, skip_special_tokens=True)
print("Outputs:\n" + 70 * '-')
for i in [0, len(outputs)-1]:
    print(f"{i}: {outputs[i]}")
    print("-" * 70)

Generate...
Outputs:
----------------------------------------------------------------------
0: Question: Where were the 2004 Olympics held?
Answer: Athens, Greece
Question: What is the longest river on the earth?
Answer: The Nile

----------------------------------------------------------------------
1: Extract the airport codes from this text.
Text: "I want a flight from New York to San Francisco."
Airport codes: JFK, SFO.
Text: "I want you to book a flight from Phoenix to Las Vegas."
Airport codes: PHX, LVG.

----------------------------------------------------------------------


Shutdown the FlexGen execution environment when done.

In [11]:
print("Shutting down...")
env.close_copy_threads()

Shutting down...
